In [89]:
import pandas as pd
import ir_datasets

qrels = (
    pd.DataFrame(ir_datasets.load("msmarco-passage/trec-dl-2020/judged").qrels_iter())
    .astype({"query_id": str, "doc_id": str})
    .drop("iteration", axis="columns")
    .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
    .merge(
        pd.read_parquet("../data/processed/trecdl-shallow-qrels-192.parquet")
        .astype({"query_id": str, "doc_id": str})
        .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
        .rename({"relevance": "relevance_adapter_192"}, axis="columns"),
        how = "left",
        on = ["query_id", "doc_id"],
    )
    .merge(
        pd.read_parquet("../data/processed/trecdl-shallow-qrels-256.parquet")
        .astype({"query_id": str, "doc_id": str})
        .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
        .rename({"relevance": "relevance_adapter_256"}, axis="columns"),
        how = "left",
        on = ["query_id", "doc_id"],
    )
    .merge(
        pd.read_parquet("../data/processed/trecdl-shallow-qrels-256.parquet")
        .astype({"query_id": str, "doc_id": str})
        .assign(relevance=0)
        .rename({"relevance": "relevance_baseline"}, axis="columns"),
        how = "left",
        on = ["query_id", "doc_id"],
    )
    .merge(
        pd.read_parquet("../data/processed/trecdl-llm-qrels.parquet")
        .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
        .query("prompt == 'few_shot_human_8'")
        .drop("prompt", axis="columns")
        .rename({"relevance": "relevance_few_shot_human_8"}, axis="columns"),
        how="left",
        on = ["query_id", "doc_id"],
    )
    .merge(
        pd.read_parquet("../data/processed/trecdl-llm-qrels.parquet")
        .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
        .query("prompt == 'few_shot_umbrela_8'")
        .drop("prompt", axis="columns")
        .rename({"relevance": "relevance_few_shot_umbrela_8"}, axis="columns"),
        how="left",
        on = ["query_id", "doc_id"],
    )
    .merge(
        pd.read_parquet("../data/processed/trecdl-llm-qrels.parquet")
        .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
        .query("prompt == 'umbrela'")
        .drop("prompt", axis="columns")
        .rename({"relevance": "relevance_umbrela"}, axis="columns"),
        how="left",
        on = ["query_id", "doc_id"],
    )
    .merge(
        pd.read_parquet("../data/processed/trecdl-llm-qrels.parquet")
        .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
        .query("prompt == 'human'")
        .drop("prompt", axis="columns")
        .rename({"relevance": "relevance_human_prompt"}, axis="columns"),
        how="left",
        on = ["query_id", "doc_id"],
    )
    .assign(
        relevance_baseline=lambda df: df["relevance_baseline"].where(~df["relevance_few_shot_umbrela_8"].isna(), df["relevance"]),
        relevance_few_shot_human_8=lambda df: df["relevance_few_shot_human_8"].where(~df["relevance_few_shot_umbrela_8"].isna(), df["relevance"]),
        relevance_umbrela=lambda df: df["relevance_umbrela"].where(~df["relevance_few_shot_umbrela_8"].isna(), df["relevance"]),
        relevance_human_prompt=lambda df: df["relevance_human_prompt"].where(~df["relevance_few_shot_umbrela_8"].isna(), df["relevance"]),
        relevance_adapter_192=lambda df: df["relevance_adapter_192"].where(~df["relevance_few_shot_umbrela_8"].isna(), df["relevance"]),
        relevance_adapter_256=lambda df: df["relevance_adapter_256"].where(~df["relevance_few_shot_umbrela_8"].isna(), df["relevance"]),
    )
    .assign(
        relevance_few_shot_umbrela_8=lambda df: df["relevance_few_shot_umbrela_8"].where(~df["relevance_few_shot_umbrela_8"].isna(), df["relevance"]),
    )
)

In [90]:
qrels

,query_id,doc_id,relevance,relevance_adapter_192,relevance_adapter_256,relevance_baseline,relevance_few_shot_human_8,relevance_few_shot_umbrela_8,relevance_umbrela,relevance_human_prompt
0,23849,1020327,1,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,23849,1034183,1,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,23849,1120730,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,23849,1139571,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,23849,1139571,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
4002948,1136962,937258,1,0.0,0.0,0.0,1.0,1.0,1.0,1.0
4002949,1136962,937258,1,0.0,0.0,0.0,1.0,1.0,1.0,1.0
4002950,1136962,937258,1,0.0,0.0,0.0,1.0,1.0,1.0,1.0
4002951,1136962,937258,1,0.0,0.0,0.0,1.0,1.0,1.0,1.0


In [92]:
import krippendorff

df_reliability = []

for c in [
    'relevance_adapter_192',
    'relevance_adapter_256',
    'relevance_baseline',
    'relevance_few_shot_umbrela_8',
    'relevance_few_shot_human_8',
    'relevance_human_prompt',
    'relevance_umbrela'
]:
    data = (
        qrels
        .dropna()
        .loc[:, ["relevance", c]]
        .values
        .T
    )
    alpha = krippendorff.alpha(reliability_data=data, level_of_measurement="nominal", value_domain=(0, 1))
    df_reliability.append({
        "qrels": c,
        "alpha": alpha,
    })

df_reliability = pd.DataFrame(df_reliability)
print(df_reliability)

                          qrels     alpha
0         relevance_adapter_192  0.559873
1         relevance_adapter_256  0.688427
2            relevance_baseline -0.466782
3  relevance_few_shot_umbrela_8  0.491042
4    relevance_few_shot_human_8  0.204431
5        relevance_human_prompt  0.158731
6             relevance_umbrela  0.459137


In [ ]:
import trectools
from glob import glob
from tqdm.notebook import tqdm

runs = [trectools.TrecRun(f) for f in glob(f"../data/external/dl20-passages-runs/*.gz")]
depths = [5, 10, 20, 50]
eval_data = []

for c in [
    'relevance_adapter_192',
    'relevance_adapter_256',
    'relevance_baseline',
    'relevance_few_shot_umbrela_8',
    'relevance_few_shot_human_8',
    'relevance_human_prompt',
    'relevance_umbrela',
    'relevance'
]:
    jdg = trectools.TrecQrel()
    jdg.qrels_data = (
        qrels
        .dropna()
        .loc[:, ["query_id", "doc_id", c]]
        .rename({"query_id": "query", "doc_id": "docid",c: "rel"}, axis="columns")
    )
    for r in tqdm(runs, desc=c):
        eval = trectools.TrecEval(r, jdg)
        eval_data.append(
            pd.concat([
                pd.concat([eval.get_ndcg(depth=k, per_query=True) for k in depths],
                    axis=1).reset_index().melt(id_vars=["query"]).rename(columns={"variable": "metric"}),
                pd.concat(
                    [eval.get_unjudged(depth=k, per_query=True) for k in depths],
                    axis=1).reset_index().melt(id_vars=["query"]).rename(columns={"variable": "metric"})
            ])
            .reset_index()
            .assign(
                system=r.get_runid(),
                source=c
            )
        )

df = pd.concat(eval_data)

In [9]:
(
    df
    .drop(["query", "index"], axis="columns")
    .dropna(subset="value")
    .astype({"value": float, "system": str, "source": str})
    .reset_index(drop=True)
    .groupby(["metric", "system", "source"])
    .mean()
    .reset_index()
    .pivot(index=["system", "metric"], columns="source", values="value")
    .reset_index()
    .set_index("system")
    .groupby("metric")
    .corr(method="spearman")
    .reset_index()
    .query("source == 'relevance'")
    .drop(["source", "relevance"], axis=1)
    .query("metric in ['NDCG@5', 'NDCG@10', 'NDCG@20', 'NDCG@50']")
    .set_index("metric")
    .transpose()
    .loc[:, ['NDCG@5', 'NDCG@10', 'NDCG@20', 'NDCG@50']]
)

source,metric,relevance_adapter_192,relevance_adapter_256,relevance_baseline,relevance_few_shot_8,relevance_gpt4_human,relevance_gpt4_umbrela
0,NDCG@10,0.877265,0.942314,NaN,0.746566,0.616868,0.329067
7,NDCG@20,0.784220,0.912624,NaN,0.802443,0.452218,0.498685
14,NDCG@5,0.881557,0.919755,NaN,0.758114,0.625528,0.382458
21,NDCG@50,0.909819,0.954763,NaN,0.729207,0.503828,0.451809
28,UNJ@10,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
35,UNJ@20,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
42,UNJ@5,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
49,UNJ@50,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
